# 01 — Associative Memory: The Unifying Abstraction

**Core claim**: Backprop, attention, and momentum are all associative memory modules.

This notebook demonstrates:
1. Classical Hopfield network — store & retrieve corrupted patterns
2. Modern (continuous) Hopfield — exponential capacity
3. Numerical verification: attention = modern Hopfield retrieval
4. Backprop weight update as outer product (associative memory write)
5. SGD momentum buffer as exponentially-weighted gradient memory

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from src.associative_memory import (
    HopfieldNetwork, ModernHopfield,
    AttentionAsAssociativeMemory,
    BackpropAsAssociativeMemory,
    MomentumAsAssociativeMemory,
)
from src.data import generate_binary_patterns, generate_letter_patterns
from src.utils import set_seed, plot_pattern
set_seed(42)

## 1. Classical Hopfield Network — Pattern Storage & Retrieval

In [ ]:
# Store 8x8 letter patterns and retrieve from corrupted versions
letters = generate_letter_patterns()
patterns = torch.stack(list(letters.values()))  # (5, 64)
names = list(letters.keys())

net = HopfieldNetwork(64)
net.store(patterns)

# Corrupt pattern A with 25% noise
query = patterns[0].clone()
flip_idx = torch.randperm(64)[:16]
query[flip_idx] *= -1
retrieved = net.retrieve(query)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
plot_pattern(patterns[0], title='Original A', ax=axes[0])
plot_pattern(query, title='Corrupted (25%)', ax=axes[1])
plot_pattern(retrieved, title='Retrieved', ax=axes[2])
plt.tight_layout()
plt.show()

overlap = (retrieved == patterns[0]).float().mean()
print(f'Retrieval overlap: {overlap:.1%}')
print(f'Energy: {net.energy(query):.2f} -> {net.energy(retrieved):.2f}')

## 2. Capacity Comparison: Classical vs. Modern Hopfield

In [ ]:
dim = 100
rng = np.random.default_rng(42)
pattern_counts = list(range(2, 60, 4))
classical_errors = []
modern_errors = []

for n_pat in pattern_counts:
    pats = generate_binary_patterns(n_pat, dim, rng=rng)
    # Classical
    cnet = HopfieldNetwork(dim)
    cnet.store(pats)
    errs = []
    for i in range(n_pat):
        ret = cnet.retrieve(pats[i])
        errs.append((ret != pats[i]).float().mean().item())
    classical_errors.append(np.mean(errs))
    # Modern
    mh = ModernHopfield()
    mh.store(pats.float())
    errs = []
    for i in range(n_pat):
        ret = mh.retrieve(pats[i].float())
        # For continuous output, check if closest stored pattern is correct
        dists = torch.cdist(ret, pats.float().unsqueeze(0) if pats.dim()==2 else pats.float())
        nearest = dists.argmin(dim=-1).item()
        errs.append(0.0 if nearest == i else 1.0)
    modern_errors.append(np.mean(errs))

plt.figure(figsize=(8, 4))
plt.plot(pattern_counts, classical_errors, 'o-', label='Classical Hopfield')
plt.plot(pattern_counts, modern_errors, 's-', label='Modern Hopfield')
plt.axvline(x=0.14 * dim, color='r', ls='--', label=f'Classical bound ({0.14*dim:.0f})')
plt.xlabel('Number of stored patterns')
plt.ylabel('Retrieval error rate')
plt.title(f'Hopfield Capacity (dim={dim})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Numerical Verification: Attention = Modern Hopfield

In [ ]:
torch.manual_seed(0)
dim = 32
X = torch.randn(10, dim)  # stored patterns
Q = torch.randn(4, dim)   # queries

# Modern Hopfield retrieval
mh = ModernHopfield()
mh.store(X)
hopfield_out = mh.retrieve(Q)

# Attention with K=V=X
attn = AttentionAsAssociativeMemory(dim)
attn.store(keys=X, values=X)
attn_out = attn.retrieve(Q)

max_diff = (hopfield_out - attn_out).abs().max().item()
print(f'Max absolute difference: {max_diff:.2e}')
assert torch.allclose(hopfield_out, attn_out, atol=1e-5)
print('PASSED: Attention and Modern Hopfield produce identical outputs!')

## 4. Backprop Weight Update as Outer Product (Associative Memory Write)

In [ ]:
# XOR task
X_xor = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_xor = torch.tensor([0,1,1,0], dtype=torch.long)

model = nn.Sequential()
model.add_module('fc1', nn.Linear(2, 8))
model.add_module('relu', nn.ReLU())
model.add_module('fc2', nn.Linear(8, 2))

wrapper = BackpropAsAssociativeMemory(model, 'fc1')

# One forward-backward pass
loss = nn.CrossEntropyLoss()(model(X_xor), y_xor)
loss.backward()

outer = wrapper.get_outer_product_update()
print(f'Outer product shape: {outer.shape}')  # (8, 2)
print(f'Actual gradient shape: {model.fc1.weight.grad.shape}')  # (8, 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(outer.numpy(), cmap='RdBu_r', aspect='auto')
axes[0].set_title('error @ activation^T\n(outer product view)')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(model.fc1.weight.grad.numpy(), cmap='RdBu_r', aspect='auto')
axes[1].set_title('Actual dW from autograd')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

wrapper.remove_hooks()

## 5. SGD Momentum as Exponentially-Weighted Gradient Memory

In [ ]:
# Train a tiny network and track momentum buffer evolution
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
mom_wrapper = MomentumAsAssociativeMemory(optimizer)

# Synthetic data: two spirals
n = 200
theta = torch.linspace(0, 4*np.pi, n)
r = torch.linspace(0.5, 2, n)
X_train = torch.stack([
    torch.cat([r * torch.cos(theta), r * torch.cos(theta + np.pi)]),
    torch.cat([r * torch.sin(theta), r * torch.sin(theta + np.pi)]),
], dim=1)
y_train = torch.cat([torch.zeros(n), torch.ones(n)]).long()

# Wrap data as infinite iterator
def data_iter():
    while True:
        idx = torch.randperm(len(X_train))[:32]
        yield X_train[idx], y_train[idx]

history = mom_wrapper.get_momentum_history(
    model, nn.CrossEntropyLoss(), data_iter(), n_steps=100
)

# Plot momentum buffer norms over time for first layer weights
buf_norms = [h[0].norm().item() for h in history]
plt.figure(figsize=(8, 3))
plt.plot(buf_norms)
plt.xlabel('Step')
plt.ylabel('Momentum buffer norm (layer 0 weights)')
plt.title('Momentum Buffer Evolution\n(exponentially-weighted gradient memory)')
plt.grid(True, alpha=0.3)
plt.show()